## Aeropulse — Gold: Flight Fact

**Purpose:** Builds `fact_flight` by joining the current batch of silver `flight` to the three gold dimensions (origin airport, destination airport, carrier) for their surrogate keys, then derives the reporting-facing measures: `total_delay_minutes`, `primary_delay_cause`, `is_delayed`, `delay_category`, `flight_status`, `route`.

**Batch parameters:** `batch_id`, `batch_year`

**Depends on:** `gold-environment`, `gold-helper` (run via `%run`)

**Reads:** `dim_origin_airport`, `dim_destination_airport`, `dim_carrier` (gold), `silver.flight` (filtered to `batch_id`)

**Writes:** `fact_flight` (merge on `flight_sk`)


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 27, Finished, Available, Finished, False)

In [2]:
batch_id = ""
batch_year = ""

StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 28, Finished, Available, Finished, False)

In [3]:
%run gold-environment

StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 29, Finished, Available, Finished, True)

In [4]:
%run gold-helper

StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 32, Finished, Available, Finished, True)

# load airport_depature, airport_destination and carrier_path df

In [5]:
airport_origin = spark.read.format('delta').load(airport_origin_path)
airport_destination = spark.read.format('delta').load(airport_destination_path)
carrier_path = spark.read.format('delta').load(carrier_path)

StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 33, Finished, Available, Finished, False)

In [6]:
# load silver flight df

flight_silver_df = spark.read.format('delta').load(flight_silver_path).filter(F.col("batch_id")==batch_id)

StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 34, Finished, Available, Finished, False)

In [7]:

fact_flight_df = (
    flight_silver_df.alias("f")
    .join(airport_origin.alias("o"),
          F.col("f.origin_airport_code") == F.col("o.origin_airport_code"), "left")
    .join(airport_destination.withColumnRenamed("airport_sk", "destination_airport_sk").alias("d"),
          F.col("f.destination_airport_code") == F.col("d.destination_airport_code"), "left")
    .join(carrier_path.alias("c"),
          F.col("f.operating_carrier_code") == F.col("c.carrier_code"), "left")
).select(
    F.col("f.*"),
    F.col("o.origin_airport_sk"),
    F.col("d.airport_destination_sk"),
    F.col("c.carrier_sk")
).drop(
    F.col("f.origin_airport_code"),
    F.col("f.destination_airport_code"),
    F.col("f.operating_carrier_code"),
    F.col("f.airport_city_name"),
)

StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 35, Finished, Available, Finished, False)

In [8]:

delay_causes = ["carrier_delay_minutes", "weather_delay_minutes", "nas_delay_minutes",
                "security_delay_minutes", "late_aircraft_delay_minutes"]

# 1. total_delay_minutes — the five cause columns are only populated when a flight was delayed,
#    so a null-safe sum gives a clean 0 for on-time flights instead of leaving it null.
fact_flight_df = fact_flight_df.withColumn(
    "total_delay_minutes",
    sum(F.coalesce(F.col(c), F.lit(0.0)) for c in delay_causes)
)

# 2. primary_delay_cause — which single cause contributed the most minutes, for cause-of-delay reporting.
max_delay = F.greatest(*[F.coalesce(F.col(c), F.lit(0.0)) for c in delay_causes])
fact_flight_df = fact_flight_df.withColumn(
    "primary_delay_cause",
    F.when(F.col("total_delay_minutes") == 0, F.lit(None))
     .when(max_delay == F.coalesce(F.col("carrier_delay_minutes"), F.lit(0.0)), F.lit("Carrier"))
     .when(max_delay == F.coalesce(F.col("weather_delay_minutes"), F.lit(0.0)), F.lit("Weather"))
     .when(max_delay == F.coalesce(F.col("nas_delay_minutes"), F.lit(0.0)), F.lit("NAS"))
     .when(max_delay == F.coalesce(F.col("security_delay_minutes"), F.lit(0.0)), F.lit("Security"))
     .otherwise(F.lit("Late Aircraft"))
)

# 3. is_delayed — the FAA/BTS standard definition: arrival at least 15 minutes late.
#    arrival_delay_minutes is already actual-minus-scheduled, straight from the source.
fact_flight_df = fact_flight_df.withColumn(
    "is_delayed", F.coalesce(F.col("arrival_delay_minutes"), F.lit(0.0)) >= 15
)

# 4. delay_category — bucketed severity for slicing in reports, cancelled/diverted handled first.
fact_flight_df = fact_flight_df.withColumn(
    "delay_category",
    F.when(F.col("is_cancelled"), F.lit("Cancelled"))
     .when(F.col("is_diverted"), F.lit("Diverted"))
     .when(F.col("arrival_delay_minutes").isNull(), F.lit("Unknown"))
     .when(F.col("arrival_delay_minutes") < 15, F.lit("On Time"))
     .when(F.col("arrival_delay_minutes") < 60, F.lit("Minor Delay"))
     .when(F.col("arrival_delay_minutes") < 180, F.lit("Moderate Delay"))
     .otherwise(F.lit("Severe Delay"))
)

# 5. flight_status — a single categorical rollup, easier to slice by than two separate booleans.
fact_flight_df = fact_flight_df.withColumn(
    "flight_status",
    F.when(F.col("is_cancelled"), F.lit("Cancelled"))
     .when(F.col("is_diverted"), F.lit("Diverted"))
     .otherwise(F.lit("Completed"))
)

# 6. route — for route-level aggregation without joining back to dim_airport twice.
fact_flight_df = fact_flight_df.withColumn(
    "route", F.concat_ws(" - ", F.col("origin_city_name"), F.col("destination_city_name"))
)

## select columns
fact_flight_df = fact_flight_df.select(
    'flight_date','tail_number','flight_number','origin_city_name','destination_city_name','departure_time','departure_delay_minutes',
    'taxi_out_minutes','taxi_in_minutes','arrival_time','arrival_delay_minutes','air_time_minutes','distance_miles',
    'batch_id','flight_date_id','is_cancelled','is_diverted', 'flight_sk', 'cancellation_code', 'origin_airport_sk','airport_destination_sk','carrier_sk','total_delay_minutes',
    'primary_delay_cause','is_delayed', 'delay_category', 'flight_status', 'route'
)


StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 36, Finished, Available, Finished, False)

In [9]:
write_to_gold(
    fact_flight_df, "fact_flight",
    "s.flight_sk = t.flight_sk",
    [c for c in fact_flight_df.columns if c != "flight_sk"]
)

StatementMeta(, fba4375c-8343-41f9-88e9-8fbb271ef29a, 37, Finished, Available, Finished, False)